<a href="https://colab.research.google.com/github/gustavo159753/Machine-learning-and-AI/blob/main/first_train_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import layers, models

In [2]:
(train_ds, validation_ds, test_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

# Definindo o tamanho padrão para o MobileNetV2
IMG_SIZE = 160

def format_example(image, label):
    image = tf.cast(image, tf.float32)
    image = (image/127.5) - 1 # Normaliza para [-1, 1]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

train = train_ds.map(format_example).shuffle(1000).batch(32)
validation = validation_ds.map(format_example).batch(32)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.TIVOB3_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


In [3]:
# Criando o modelo base
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

base_model = tf.keras.applications.MobileNetV2 (input_shape=IMG_SHAPE, include_top=False, weights='imagenet')

# CONGELANDO o modelo base (Não queremos treinar as camadas que já sabem o que é um olho ou uma orelha)
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [4]:
model = models.Sequential([base_model, layers.GlobalAveragePooling2D(), layers.Dense(1, activation='sigmoid') # 1 neurônio pois é binário (Gato ou Cão)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

In [5]:
history = model.fit(train, epochs=5, validation_data=validation)

Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 470s 792ms/step - accuracy: 0.9107 - loss: 0.2565 - val_accuracy: 0.9716 - val_loss: 0.1159
Epoch 2/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 433s 740ms/step - accuracy: 0.9724 - loss: 0.0954 - val_accuracy: 0.9776 - val_loss: 0.0780
Epoch 3/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 471s 807ms/step - accuracy: 0.9771 - loss: 0.0721 - val_accuracy: 0.9811 - val_loss: 0.0649
Epoch 4/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 436s 744ms/step - accuracy: 0.9790 - loss: 0.0618 - val_accuracy: 0.9819 - val_loss: 0.0583
Epoch 5/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 427s 731ms/step - accuracy: 0.9807 - loss: 0.0559 - val_accuracy: 0.9824 - val_loss: 0.0543
